In [1]:
!pip install catboost

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
import pandas as pd
import numpy as np

In [3]:
data_folder = '/kaggle/input/antimicrobial-resistance-prediction-from-maldi-tof'

In [4]:
train = pd.read_csv(f"{data_folder}/train.csv")
test  = pd.read_csv(f"{data_folder}/test.csv")
sub   = pd.read_csv(f"{data_folder}/sample_submission.csv")

print(train.shape)
print(test.shape)
print(sub.shape)
train.head(2)


(3360, 6010)
(1000, 6002)
(1000, 9)


,sample_id,species_id,maldi_feature_0,maldi_feature_1,maldi_feature_2,maldi_feature_3,maldi_feature_4,maldi_feature_5,maldi_feature_6,maldi_feature_7,...,maldi_feature_5998,maldi_feature_5999,Ampicillin,Levofloxacin,Ciprofloxacin,Imipenem,Amoxicillin_Clavulanic_acid,Ertapenem,Cefotaxime,Cefuroxime
0,SAMPLE_00000,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SAMPLE_00001,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [22]:
# Fold-safe conservative pseudo-labeling for ONE target: Cefuroxime
# - Teacher is trained ONLY on labeled training-fold data
# - Pseudo-labels are created ONLY from teacher predictions on unlabeled rows
# - Student is trained on (labeled train-fold + pseudo-labeled) and evaluated on labeled val-fold
# - Validation set is NEVER pseudo-labeled, NEVER augmented

TARGET = "Cefotaxime"

TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime",
]

# Features: species_id + MALDI features
maldi_cols = [c for c in train.columns if c.startswith("maldi_feature_")]
feature_cols = ["species_id"] + maldi_cols

cat_features = ["species_id"]

# CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_cb(seed=42, use_gpu=False):
    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=seed,
        verbose=False,
        od_type="Iter",
        od_wait=200,
    )
    if use_gpu:
        params.update(task_type="GPU")
    return CatBoostClassifier(**params)

# Conservative thresholds
THR_POS = 0.98
THR_NEG = 0.02

# Strongly recommended for trustworthy deltas:
# GPU CatBoost can be slightly non-deterministic -> use CPU for A/B comparison first.
USE_GPU = True

# ----------------------------
# Prepare labeled / unlabeled
# ----------------------------
if TARGET not in train.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Available: {train.columns.tolist()[:30]} ...")

labeled_df = train.dropna(subset=[TARGET]).copy()
unlabeled_df = train[train[TARGET].isna()].copy()

X_lab = labeled_df[feature_cols]
y_lab = labeled_df[TARGET].astype(int)

X_unlab = unlabeled_df[feature_cols]  # no labels

print(f"Target: {TARGET}")
print(f"Labeled:   {len(labeled_df)}")
print(f"Unlabeled: {len(unlabeled_df)}")
print(f"Features:  {len(feature_cols)} (cat: {cat_features})")

# ----------------------------
# Baseline CV (no pseudo-labels) — same model/params
# ----------------------------
oof_base = np.zeros(len(labeled_df), dtype=float)
fold_auc_base = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    model = make_cb(seed=42 + fold, use_gpu=USE_GPU)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    p_va = model.predict_proba(X_va)[:, 1]
    oof_base[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_base.append(auc)
    print(f"[Baseline] Fold {fold} AUC: {auc:.5f}")

auc_base = roc_auc_score(y_lab, oof_base)
print(f"[Baseline] OOF AUC: {auc_base:.5f}")
print(f"[Baseline] Mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")

# ----------------------------
# Fold-safe pseudo-labeling CV
# ----------------------------
oof_pl = np.zeros(len(labeled_df), dtype=float)
fold_auc_pl = []
pseudo_counts = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    # 1) Teacher trained ONLY on labeled training fold
    teacher = make_cb(seed=1000 + fold, use_gpu=USE_GPU)
    teacher.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 2) Teacher predicts on UNLABELED pool
    if len(X_unlab) > 0:
        p_unlab = teacher.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab >= THR_POS
        sel_neg = p_unlab <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl = X_unlab.loc[sel].copy()
        y_pl = pd.Series(np.where(p_unlab[sel] >= THR_POS, 1, 0), index=X_pl.index)

        n_pos = int(sel_pos.sum())
        n_neg = int(sel_neg.sum())
        n_tot = int(sel.sum())
    else:
        X_pl = X_unlab.iloc[0:0].copy()
        y_pl = pd.Series([], dtype=int)
        n_pos = n_neg = n_tot = 0

    pseudo_counts.append((n_tot, n_pos, n_neg))
    print(f"[PL] Fold {fold} pseudo-labels: total={n_tot}, pos={n_pos}, neg={n_neg}")

    # 3) Student trained on (labeled train fold + fold-specific pseudo-labels)
    if n_tot > 0:
        X_tr_aug = pd.concat([X_tr, X_pl], axis=0)
        y_tr_aug = pd.concat([y_tr, y_pl], axis=0).astype(int)
    else:
        X_tr_aug, y_tr_aug = X_tr, y_tr

    student = make_cb(seed=2000 + fold, use_gpu=USE_GPU)
    student.fit(
        X_tr_aug, y_tr_aug,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 4) Evaluate ONLY on labeled validation fold
    p_va = student.predict_proba(X_va)[:, 1]
    oof_pl[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_pl.append(auc)
    print(f"[PL] Fold {fold} AUC: {auc:.5f}")

auc_pl = roc_auc_score(y_lab, oof_pl)

print("\n========== Summary ==========")
print(f"Target: {TARGET}")
print(f"Thresholds: pos>={THR_POS}, neg<={THR_NEG}")
print("Pseudo-label counts per fold (total, pos, neg):")
for i, (t, p, n) in enumerate(pseudo_counts, 1):
    print(f"  Fold {i}: ({t}, {p}, {n})")

print(f"\nBaseline OOF AUC: {auc_base:.5f}")
print(f"PL OOF AUC:       {auc_pl:.5f}")
print(f"Delta:            {auc_pl - auc_base:+.5f}")

print(f"\nBaseline mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")
print(f"PL mean fold AUC:       {np.mean(fold_auc_pl):.5f} ± {np.std(fold_auc_pl):.5f}")

# Decision rule (strict)
KEEP_IF_DELTA_AT_LEAST = 0.002
MAX_FOLD_DROP_ALLOWED = 0.01
fold_deltas = np.array(fold_auc_pl) - np.array(fold_auc_base)
worst_drop = fold_deltas.min()
keep = (auc_pl - auc_base) >= KEEP_IF_DELTA_AT_LEAST and worst_drop >= -MAX_FOLD_DROP_ALLOWED

print(f"\nDecision: {'KEEP' if keep else 'DISCARD'} "
      f"(needs delta >= {KEEP_IF_DELTA_AT_LEAST} and worst fold drop >= -{MAX_FOLD_DROP_ALLOWED})")

# ----------------------------
# OPTIONAL: If KEEP, train final model and make a Cefuroxime-only submission column
# (This is NOT your full competition submission; it's just to inspect the effect.)
# ----------------------------
if keep:
    print("\n[Final] Training teacher on ALL labeled data to pseudo-label unlabeled, then training final model...")

    # Teacher on all labeled
    teacher_full = make_cb(seed=4242, use_gpu=USE_GPU)
    teacher_full.fit(
        X_lab, y_lab,
        cat_features=cat_features,
        verbose=False
    )

    if len(X_unlab) > 0:
        p_unlab_full = teacher_full.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab_full >= THR_POS
        sel_neg = p_unlab_full <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl_full = X_unlab.loc[sel].copy()
        y_pl_full = pd.Series(np.where(p_unlab_full[sel] >= THR_POS, 1, 0), index=X_pl_full.index).astype(int)

        print(f"[Final] Pseudo-labels added: {sel.sum()} (pos={sel_pos.sum()}, neg={sel_neg.sum()})")

        X_train_final = pd.concat([X_lab, X_pl_full], axis=0)
        y_train_final = pd.concat([y_lab, y_pl_full], axis=0).astype(int)
    else:
        X_train_final = X_lab
        y_train_final = y_lab

    final_model = make_cb(seed=5252, use_gpu=USE_GPU)
    final_model.fit(
        X_train_final, y_train_final,
        cat_features=cat_features,
        verbose=False
    )

    # Predict on test
    p_test = final_model.predict_proba(test[feature_cols])[:, 1]

    out = pd.DataFrame({"sample_id": test["sample_id"].values, TARGET: p_test})
    out.to_csv("/kaggle/working/submission_cefuroxime_pl_only.csv", index=False)
    print("Wrote: submission_cefuroxime_pl_only.csv")


Target: Cefotaxime
Labeled:   3357
Unlabeled: 3
Features:  6001 (cat: ['species_id'])


Default metric period is 5 because AUC is/are not implemented for GPU


[Baseline] Fold 1 AUC: 0.92692


Default metric period is 5 because AUC is/are not implemented for GPU


[Baseline] Fold 2 AUC: 0.92579


...

[PL] Fold 5 AUC: 0.93898

========== Summary ==========
Target: Cefotaxime
Thresholds: pos>=0.98, neg<=0.02
Pseudo-label counts per fold (total, pos, neg):
  Fold 1: (0, 0, 0)
  Fold 2: (0, 0, 0)
  Fold 3: (0, 0, 0)
  Fold 4: (0, 0, 0)
  Fold 5: (0, 0, 0)

Baseline OOF AUC: 0.93144
PL OOF AUC:       0.93504
Delta:            +0.00359

Baseline mean fold AUC: 0.93259 ± 0.00569
PL mean fold AUC:       0.93553 ± 0.00739

Decision: KEEP (needs delta >= 0.002 and worst fold drop >= -0.01)

[Final] Training teacher on ALL labeled data to pseudo-label unlabeled, then training final model...


Default metric period is 5 because AUC is/are not implemented for GPU


[Final] Pseudo-labels added: 0 (pos=0, neg=0)


Default metric period is 5 because AUC is/are not implemented for GPU


Wrote: submission_cefuroxime_pl_only.csv


In [5]:
# Fold-safe conservative pseudo-labeling for ONE target: Cefuroxime
# - Teacher is trained ONLY on labeled training-fold data
# - Pseudo-labels are created ONLY from teacher predictions on unlabeled rows
# - Student is trained on (labeled train-fold + pseudo-labeled) and evaluated on labeled val-fold
# - Validation set is NEVER pseudo-labeled, NEVER augmented

TARGET = "Ampicillin"

TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime",
]

# Features: species_id + MALDI features
maldi_cols = [c for c in train.columns if c.startswith("maldi_feature_")]
feature_cols = ["species_id"] + maldi_cols

cat_features = ["species_id"]

# CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_cb(seed=42, use_gpu=False):
    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=seed,
        verbose=False,
        od_type="Iter",
        od_wait=200,
    )
    if use_gpu:
        params.update(task_type="GPU")
    return CatBoostClassifier(**params)

# Conservative thresholds
THR_POS = 0.98
THR_NEG = 0.02

# Strongly recommended for trustworthy deltas:
# GPU CatBoost can be slightly non-deterministic -> use CPU for A/B comparison first.
USE_GPU = False

# ----------------------------
# Prepare labeled / unlabeled
# ----------------------------
if TARGET not in train.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Available: {train.columns.tolist()[:30]} ...")

labeled_df = train.dropna(subset=[TARGET]).copy()
unlabeled_df = train[train[TARGET].isna()].copy()

X_lab = labeled_df[feature_cols]
y_lab = labeled_df[TARGET].astype(int)

X_unlab = unlabeled_df[feature_cols]  # no labels

print(f"Target: {TARGET}")
print(f"Labeled:   {len(labeled_df)}")
print(f"Unlabeled: {len(unlabeled_df)}")
print(f"Features:  {len(feature_cols)} (cat: {cat_features})")

# ----------------------------
# Baseline CV (no pseudo-labels) — same model/params
# ----------------------------
oof_base = np.zeros(len(labeled_df), dtype=float)
fold_auc_base = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    model = make_cb(seed=42 + fold, use_gpu=USE_GPU)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    p_va = model.predict_proba(X_va)[:, 1]
    oof_base[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_base.append(auc)
    print(f"[Baseline] Fold {fold} AUC: {auc:.5f}")

auc_base = roc_auc_score(y_lab, oof_base)
print(f"[Baseline] OOF AUC: {auc_base:.5f}")
print(f"[Baseline] Mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")

# ----------------------------
# Fold-safe pseudo-labeling CV
# ----------------------------
oof_pl = np.zeros(len(labeled_df), dtype=float)
fold_auc_pl = []
pseudo_counts = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    # 1) Teacher trained ONLY on labeled training fold
    teacher = make_cb(seed=1000 + fold, use_gpu=USE_GPU)
    teacher.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 2) Teacher predicts on UNLABELED pool
    if len(X_unlab) > 0:
        p_unlab = teacher.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab >= THR_POS
        sel_neg = p_unlab <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl = X_unlab.loc[sel].copy()
        y_pl = pd.Series(np.where(p_unlab[sel] >= THR_POS, 1, 0), index=X_pl.index)

        n_pos = int(sel_pos.sum())
        n_neg = int(sel_neg.sum())
        n_tot = int(sel.sum())
    else:
        X_pl = X_unlab.iloc[0:0].copy()
        y_pl = pd.Series([], dtype=int)
        n_pos = n_neg = n_tot = 0

    pseudo_counts.append((n_tot, n_pos, n_neg))
    print(f"[PL] Fold {fold} pseudo-labels: total={n_tot}, pos={n_pos}, neg={n_neg}")

    # 3) Student trained on (labeled train fold + fold-specific pseudo-labels)
    if n_tot > 0:
        X_tr_aug = pd.concat([X_tr, X_pl], axis=0)
        y_tr_aug = pd.concat([y_tr, y_pl], axis=0).astype(int)
    else:
        X_tr_aug, y_tr_aug = X_tr, y_tr

    student = make_cb(seed=2000 + fold, use_gpu=USE_GPU)
    student.fit(
        X_tr_aug, y_tr_aug,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 4) Evaluate ONLY on labeled validation fold
    p_va = student.predict_proba(X_va)[:, 1]
    oof_pl[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_pl.append(auc)
    print(f"[PL] Fold {fold} AUC: {auc:.5f}")

auc_pl = roc_auc_score(y_lab, oof_pl)

print("\n========== Summary ==========")
print(f"Target: {TARGET}")
print(f"Thresholds: pos>={THR_POS}, neg<={THR_NEG}")
print("Pseudo-label counts per fold (total, pos, neg):")
for i, (t, p, n) in enumerate(pseudo_counts, 1):
    print(f"  Fold {i}: ({t}, {p}, {n})")

print(f"\nBaseline OOF AUC: {auc_base:.5f}")
print(f"PL OOF AUC:       {auc_pl:.5f}")
print(f"Delta:            {auc_pl - auc_base:+.5f}")

print(f"\nBaseline mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")
print(f"PL mean fold AUC:       {np.mean(fold_auc_pl):.5f} ± {np.std(fold_auc_pl):.5f}")

# Decision rule (strict)
KEEP_IF_DELTA_AT_LEAST = 0.002
MAX_FOLD_DROP_ALLOWED = 0.01
fold_deltas = np.array(fold_auc_pl) - np.array(fold_auc_base)
worst_drop = fold_deltas.min()
keep = (auc_pl - auc_base) >= KEEP_IF_DELTA_AT_LEAST and worst_drop >= -MAX_FOLD_DROP_ALLOWED

print(f"\nDecision: {'KEEP' if keep else 'DISCARD'} "
      f"(needs delta >= {KEEP_IF_DELTA_AT_LEAST} and worst fold drop >= -{MAX_FOLD_DROP_ALLOWED})")

# ----------------------------
# OPTIONAL: If KEEP, train final model and make a Cefuroxime-only submission column
# (This is NOT your full competition submission; it's just to inspect the effect.)
# ----------------------------
if keep:
    print("\n[Final] Training teacher on ALL labeled data to pseudo-label unlabeled, then training final model...")

    # Teacher on all labeled
    teacher_full = make_cb(seed=4242, use_gpu=USE_GPU)
    teacher_full.fit(
        X_lab, y_lab,
        cat_features=cat_features,
        verbose=False
    )

    if len(X_unlab) > 0:
        p_unlab_full = teacher_full.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab_full >= THR_POS
        sel_neg = p_unlab_full <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl_full = X_unlab.loc[sel].copy()
        y_pl_full = pd.Series(np.where(p_unlab_full[sel] >= THR_POS, 1, 0), index=X_pl_full.index).astype(int)

        print(f"[Final] Pseudo-labels added: {sel.sum()} (pos={sel_pos.sum()}, neg={sel_neg.sum()})")

        X_train_final = pd.concat([X_lab, X_pl_full], axis=0)
        y_train_final = pd.concat([y_lab, y_pl_full], axis=0).astype(int)
    else:
        X_train_final = X_lab
        y_train_final = y_lab

    final_model = make_cb(seed=5252, use_gpu=USE_GPU)
    final_model.fit(
        X_train_final, y_train_final,
        cat_features=cat_features,
        verbose=False
    )

    # Predict on test
    p_test = final_model.predict_proba(test[feature_cols])[:, 1]

    out = pd.DataFrame({"sample_id": test["sample_id"].values, TARGET: p_test})
    out.to_csv("/kaggle/working/submission_cefuroxime_pl_only.csv", index=False)
    print("Wrote: submission_cefuroxime_pl_only.csv")


Target: Ampicillin
Labeled:   3332
Unlabeled: 28
Features:  6001 (cat: ['species_id'])
[Baseline] Fold 1 AUC: 0.92709
[Baseline] Fold 2 AUC: 0.93479
[Baseline] Fold 3 AUC: 0.94030
[Baseline] Fold 4 AUC: 0.94289
[Baseline] Fold 5 AUC: 0.92810
[Baseline] OOF AUC: 0.90169
[Baseline] Mean fold AUC: 0.93463 ± 0.00632
[PL] Fold 1 pseudo-labels: total=0, pos=0, neg=0
[PL] Fold 1 AUC: 0.92616
[PL] Fold 2 pseudo-labels: total=0, pos=0, neg=0
[PL] Fold 2 AUC: 0.93628
[PL] Fold 3 pseudo-labels: total=0, pos=0, neg=0
[PL] Fold 3 AUC: 0.94307
[PL] Fold 4 pseudo-labels: total=0, pos=0, neg=0
[PL] Fold 4 AUC: 0.94175
[PL] Fold 5 pseudo-labels: total=0, pos=0, neg=0
[PL] Fold 5 AUC: 0.92741

========== Summary ==========
Target: Ampicillin
Thresholds: pos>=0.98, neg<=0.02
Pseudo-label counts per fold (total, pos, neg):
  Fold 1: (0, 0, 0)
  Fold 2: (0, 0, 0)
  Fold 3: (0, 0, 0)
  Fold 4: (0, 0, 0)
  Fold 5: (0, 0, 0)

Baseline OOF AUC: 0.90169
PL OOF AUC:       0.93281
Delta:            +0.03112

Base

In [6]:
# Fold-safe conservative pseudo-labeling for ONE target: Cefuroxime
# - Teacher is trained ONLY on labeled training-fold data
# - Pseudo-labels are created ONLY from teacher predictions on unlabeled rows
# - Student is trained on (labeled train-fold + pseudo-labeled) and evaluated on labeled val-fold
# - Validation set is NEVER pseudo-labeled, NEVER augmented

TARGET = "Amoxicillin_Clavulanic_acid"

TARGETS = [
    "Ampicillin",
    "Levofloxacin",
    "Ciprofloxacin",
    "Imipenem",
    "Amoxicillin_Clavulanic_acid",
    "Ertapenem",
    "Cefotaxime",
    "Cefuroxime",
]

# Features: species_id + MALDI features
maldi_cols = [c for c in train.columns if c.startswith("maldi_feature_")]
feature_cols = ["species_id"] + maldi_cols

cat_features = ["species_id"]

# CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_cb(seed=42, use_gpu=False):
    params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=seed,
        verbose=False,
        od_type="Iter",
        od_wait=200,
    )
    if use_gpu:
        params.update(task_type="GPU")
    return CatBoostClassifier(**params)

# Conservative thresholds
THR_POS = 0.98
THR_NEG = 0.02

# Strongly recommended for trustworthy deltas:
# GPU CatBoost can be slightly non-deterministic -> use CPU for A/B comparison first.
USE_GPU = True

# ----------------------------
# Prepare labeled / unlabeled
# ----------------------------
if TARGET not in train.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Available: {train.columns.tolist()[:30]} ...")

labeled_df = train.dropna(subset=[TARGET]).copy()
unlabeled_df = train[train[TARGET].isna()].copy()

X_lab = labeled_df[feature_cols]
y_lab = labeled_df[TARGET].astype(int)

X_unlab = unlabeled_df[feature_cols]  # no labels

print(f"Target: {TARGET}")
print(f"Labeled:   {len(labeled_df)}")
print(f"Unlabeled: {len(unlabeled_df)}")
print(f"Features:  {len(feature_cols)} (cat: {cat_features})")

# ----------------------------
# Baseline CV (no pseudo-labels) — same model/params
# ----------------------------
oof_base = np.zeros(len(labeled_df), dtype=float)
fold_auc_base = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    model = make_cb(seed=42 + fold, use_gpu=USE_GPU)
    model.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    p_va = model.predict_proba(X_va)[:, 1]
    oof_base[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_base.append(auc)
    print(f"[Baseline] Fold {fold} AUC: {auc:.5f}")

auc_base = roc_auc_score(y_lab, oof_base)
print(f"[Baseline] OOF AUC: {auc_base:.5f}")
print(f"[Baseline] Mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")

# ----------------------------
# Fold-safe pseudo-labeling CV
# ----------------------------
oof_pl = np.zeros(len(labeled_df), dtype=float)
fold_auc_pl = []
pseudo_counts = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_lab, y_lab), start=1):
    X_tr, X_va = X_lab.iloc[tr_idx], X_lab.iloc[va_idx]
    y_tr, y_va = y_lab.iloc[tr_idx], y_lab.iloc[va_idx]

    # 1) Teacher trained ONLY on labeled training fold
    teacher = make_cb(seed=1000 + fold, use_gpu=USE_GPU)
    teacher.fit(
        X_tr, y_tr,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 2) Teacher predicts on UNLABELED pool
    if len(X_unlab) > 0:
        p_unlab = teacher.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab >= THR_POS
        sel_neg = p_unlab <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl = X_unlab.loc[sel].copy()
        y_pl = pd.Series(np.where(p_unlab[sel] >= THR_POS, 1, 0), index=X_pl.index)

        n_pos = int(sel_pos.sum())
        n_neg = int(sel_neg.sum())
        n_tot = int(sel.sum())
    else:
        X_pl = X_unlab.iloc[0:0].copy()
        y_pl = pd.Series([], dtype=int)
        n_pos = n_neg = n_tot = 0

    pseudo_counts.append((n_tot, n_pos, n_neg))
    print(f"[PL] Fold {fold} pseudo-labels: total={n_tot}, pos={n_pos}, neg={n_neg}")

    # 3) Student trained on (labeled train fold + fold-specific pseudo-labels)
    if n_tot > 0:
        X_tr_aug = pd.concat([X_tr, X_pl], axis=0)
        y_tr_aug = pd.concat([y_tr, y_pl], axis=0).astype(int)
    else:
        X_tr_aug, y_tr_aug = X_tr, y_tr

    student = make_cb(seed=2000 + fold, use_gpu=USE_GPU)
    student.fit(
        X_tr_aug, y_tr_aug,
        eval_set=(X_va, y_va),
        cat_features=cat_features,
        use_best_model=True,
        verbose=False
    )

    # 4) Evaluate ONLY on labeled validation fold
    p_va = student.predict_proba(X_va)[:, 1]
    oof_pl[va_idx] = p_va

    auc = roc_auc_score(y_va, p_va)
    fold_auc_pl.append(auc)
    print(f"[PL] Fold {fold} AUC: {auc:.5f}")

auc_pl = roc_auc_score(y_lab, oof_pl)

print("\n========== Summary ==========")
print(f"Target: {TARGET}")
print(f"Thresholds: pos>={THR_POS}, neg<={THR_NEG}")
print("Pseudo-label counts per fold (total, pos, neg):")
for i, (t, p, n) in enumerate(pseudo_counts, 1):
    print(f"  Fold {i}: ({t}, {p}, {n})")

print(f"\nBaseline OOF AUC: {auc_base:.5f}")
print(f"PL OOF AUC:       {auc_pl:.5f}")
print(f"Delta:            {auc_pl - auc_base:+.5f}")

print(f"\nBaseline mean fold AUC: {np.mean(fold_auc_base):.5f} ± {np.std(fold_auc_base):.5f}")
print(f"PL mean fold AUC:       {np.mean(fold_auc_pl):.5f} ± {np.std(fold_auc_pl):.5f}")

# Decision rule (strict)
KEEP_IF_DELTA_AT_LEAST = 0.002
MAX_FOLD_DROP_ALLOWED = 0.01
fold_deltas = np.array(fold_auc_pl) - np.array(fold_auc_base)
worst_drop = fold_deltas.min()
keep = (auc_pl - auc_base) >= KEEP_IF_DELTA_AT_LEAST and worst_drop >= -MAX_FOLD_DROP_ALLOWED

print(f"\nDecision: {'KEEP' if keep else 'DISCARD'} "
      f"(needs delta >= {KEEP_IF_DELTA_AT_LEAST} and worst fold drop >= -{MAX_FOLD_DROP_ALLOWED})")

# ----------------------------
# OPTIONAL: If KEEP, train final model and make a Cefuroxime-only submission column
# (This is NOT your full competition submission; it's just to inspect the effect.)
# ----------------------------
if keep:
    print("\n[Final] Training teacher on ALL labeled data to pseudo-label unlabeled, then training final model...")

    # Teacher on all labeled
    teacher_full = make_cb(seed=4242, use_gpu=USE_GPU)
    teacher_full.fit(
        X_lab, y_lab,
        cat_features=cat_features,
        verbose=False
    )

    if len(X_unlab) > 0:
        p_unlab_full = teacher_full.predict_proba(X_unlab)[:, 1]
        sel_pos = p_unlab_full >= THR_POS
        sel_neg = p_unlab_full <= THR_NEG
        sel = sel_pos | sel_neg

        X_pl_full = X_unlab.loc[sel].copy()
        y_pl_full = pd.Series(np.where(p_unlab_full[sel] >= THR_POS, 1, 0), index=X_pl_full.index).astype(int)

        print(f"[Final] Pseudo-labels added: {sel.sum()} (pos={sel_pos.sum()}, neg={sel_neg.sum()})")

        X_train_final = pd.concat([X_lab, X_pl_full], axis=0)
        y_train_final = pd.concat([y_lab, y_pl_full], axis=0).astype(int)
    else:
        X_train_final = X_lab
        y_train_final = y_lab

    final_model = make_cb(seed=5252, use_gpu=USE_GPU)
    final_model.fit(
        X_train_final, y_train_final,
        cat_features=cat_features,
        verbose=False
    )

    # Predict on test
    p_test = final_model.predict_proba(test[feature_cols])[:, 1]

    out = pd.DataFrame({"sample_id": test["sample_id"].values, TARGET: p_test})
    out.to_csv("/kaggle/working/submission_cefuroxime_pl_only.csv", index=False)
    print("Wrote: submission_cefuroxime_pl_only.csv")


Target: Amoxicillin_Clavulanic_acid
Labeled:   1921
Unlabeled: 1439
Features:  6001 (cat: ['species_id'])
